# Estratégia de Infraestrutura e Otimização - Centauro ERP

Este documento registra as decisões técnicas tomadas para a futura hospedagem na nuvem e as diretrizes de otimização de performance do sistema.

---

## 1. Arquitetura de Deploy (Hospedagem)

Para garantir escalabilidade, facilidade de manutenção e baixo custo inicial, o sistema será hospedado utilizando uma arquitetura separada (**Frontend** e **Backend** desacoplados).

### A. Frontend (React/Vite)
* **Provedor Escolhido:** [Vercel](https://vercel.com/) (ou Netlify).
* **Motivo:** Otimização nativa para React, CDN global (velocidade), SSL gratuito e integração contínua (CI/CD) automática com o GitHub.
* **Custo Estimado:** Gratuito (Tier Hobby).

### B. Backend (FastAPI) + Banco de Dados
* **Provedor Escolhido:** [Railway](https://railway.app/) (ou Render).
* **Tecnologia:** Container Docker.
* **Banco de Dados:** PostgreSQL (Migrar do SQLite local para Postgres na nuvem).
* **Motivo:** Facilidade de configuração (PaaS), gestão automática de segredos (`.env`) e persistência de dados.
* **Custo Estimado:** ~$5 a $15 USD/mês (Plano Hobby/Starter).

### Pré-requisitos para Deploy
1.  **Dockerização:** O Backend deve possuir um `Dockerfile` na raiz configurado para instalar dependências do sistema (`gcc`, `libpq-dev`) e bibliotecas Python.
2.  **Variáveis de Ambiente:** As credenciais não podem estar no código (`.env` local). Devem ser injetadas pela plataforma de nuvem.
3.  **Banco de Dados:** Ajustar a `connection string` do SQLAlchemy para aceitar a URL do PostgreSQL (`postgresql://...`) em vez do arquivo local.

---

## 2. Diretrizes de Otimização de Performance

Para garantir que o sistema suporte 50+ usuários simultâneos sem lentidão, devemos seguir as práticas abaixo durante o desenvolvimento.

### A. Otimização de Backend (Python/SQLAlchemy)
*O maior gargalo de performance geralmente ocorre aqui.*

1.  **Evitar o Problema N+1 (Crítico):**
    * *O que é:* Fazer 1 consulta para listar projetos e mais 50 consultas para buscar o nome do coordenador de cada um.
    * *Solução:* Usar **Eager Loading** (`joinedload` no SQLAlchemy) para trazer os dados relacionados em uma única query SQL.
    * *Regra:* Sempre revisar os endpoints de listagem (`GET /items`).

2.  **Índices no Banco de Dados:**
    * Criar índices (`indexes`) nas colunas usadas frequentemente em filtros de busca (ex: `cpf`, `numero_contrato`, `email`).
    * *Impacto:* Transforma buscas que levariam segundos em milissegundos.

3.  **Payloads Enxutos (DTOs):**
    * Não retornar o objeto inteiro do banco para o Frontend.
    * Usar **Pydantic Schemas** (`ResponseModels`) para filtrar dados desnecessários (ex: não enviar o histórico completo de logs na listagem simples de projetos).

### B. Otimização de Frontend (React)
*Foco na experiência do usuário e economia de processamento no navegador.*

1.  **Debounce na Busca:**
    * Ao digitar em campos de pesquisa, aguardar o usuário parar de digitar (ex: 300ms) antes de disparar a requisição para a API. Evita sobrecarga de chamadas inúteis.

2.  **Code Splitting (Lazy Loading):**
    * Carregar módulos pesados (Gráficos, Mapas, Editor de Texto) apenas quando o usuário acessar a rota específica.
    * Uso de `React.lazy()` e `Suspense`.

3.  **Memoização:**
    * Usar `useMemo` para cálculos pesados e `useCallback` para funções passadas como props, evitando renderizações desnecessárias de componentes filhos.

---

## 3. Checklist de Implementação Futura

- [ ] Criar `Dockerfile` na raiz do diretório `backend`.
- [ ] Criar arquivo `railway.json` ou `render.yaml` (se necessário).
- [ ] Testar conexão localmente usando um container PostgreSQL (para simular a produção).
- [ ] Configurar lista de CORS no FastAPI para aceitar o domínio da Vercel.
- [ ] Revisar endpoints principais buscando falhas de N+1.